In [4]:
import pandas as pd
import os
import joblib
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

print("=== MEMULAI FASE 3: PERSONALIZED INTERVENTION (CLUSTERING) ===\n")

# ==========================================
# LANGKAH 1: Load Data
# ==========================================
df = pd.read_csv('university_student_stress_dataset.csv')

# ==========================================
# LANGKAH 2: Memilih Fitur Perilaku (Behavioral Profiling)
# ==========================================
# Bedasarkan Feature Importance dari Fase 2, kita pilih metrik gaya hidup/akademik utama 
# untuk membuat "Interaction Labels" (Digital Engagement, Academic Load, Sleep Qualityr)
# Cukup gunakan 3 fitur ini untuk klasterisasi yang lebih tajam
clustering_features = ['Screen_Time', 'Assignment_Load', 'Sleep_Hours']

X_cluster = df[clustering_features]

# ==========================================
# LANGKAH 3: Scaling Data Khusus Clustering
# ==========================================
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

# Simpan scaler untuk clustering agar bisa digunakan di deployment
os.makedirs('models', exist_ok=True)
joblib.dump(scaler_cluster, 'models/scaler_cluster.pkl')

# ==========================================
# LANGKAH 4: Pemodelan K-Means Clustering
# ==========================================
# Kita bagi mahasiswa menjadi 3 segmen/profil perilaku
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_cluster_scaled)

# Masukkan label klaster kembali ke dataframe untuk analisis
df['Behavioral_Cluster'] = cluster_labels

# Simpan model K-Means untuk deployment
joblib.dump(kmeans, 'models/kmeans_model.pkl')
print("[INFO] Model K-Means dan Scaler berhasil diekspor ke folder 'models/'.\n")

# ==========================================
# LANGKAH 5: Evaluasi Metrik (Silhouette Coefficient)
# ==========================================
# Sesuai proposal, mengukur seberapa baik data terpisah antar klaster
sil_score = silhouette_score(X_cluster_scaled, cluster_labels)
print("=== HASIL EVALUASI CLUSTERING ===")
print(f"Silhouette Coefficient : {sil_score:.4f} (Mendekati 1 lebih baik)\n")

# ==========================================
# LANGKAH 6: Profiling Karakteristik Klaster
# ==========================================
print("=== PROFIL BEHAVIORAL CLUSTER (RATA-RATA NILAI FITUR) ===")
# Mengembalikan nilai yang di-scale ke nilai aslinya untuk diinterpretasikan
cluster_centers_original = scaler_cluster.inverse_transform(kmeans.cluster_centers_)
cluster_df = pd.DataFrame(cluster_centers_original, columns=clustering_features)
cluster_df.index.name = 'Cluster'
print(cluster_df.round(2).to_string())

# ==========================================
# LANGKAH 7: Logika Recommender System (Content-Based Filtering) - REVISI 3 FITUR
# ==========================================
print("\n=== SISTEM REKOMENDASI INTERVENSI PERSONAL ===")
def get_recommendation(cluster_id):
    if cluster_id == 0:
        return (
            "Kategori: Kurang Tidur & Paparan Layar Tinggi\n"
            "Intervensi: Jam tidur Anda sangat kurang (di bawah 5 jam) diiringi waktu layar yang tinggi. "
            "Saran: Kurangi menatap layar perangkat 1-2 jam sebelum tidur. Prioritaskan perbaikan "
            "'Sleep Hygiene' agar mencapai minimal 6-7 jam tidur untuk menghindari kelelahan fisik (burnout)."
        )
    elif cluster_id == 1:
        return (
            "Kategori: Gaya Hidup Akademik Seimbang\n"
            "Intervensi: Hebat! Anda memiliki manajemen waktu layar yang sangat rendah dan jam tidur yang ideal. "
            "Saran: Pertahankan rutinitas sehat ini. Teruskan manajemen waktu yang baik antara istirahat dan beban tugas."
        )
    elif cluster_id == 2:
        return (
            "Kategori: Jam Tidur Tinggi & Waktu Layar Berlebih\n"
            "Intervensi: Meskipun Anda memiliki jam tidur yang sangat cukup (di atas 8 jam), waktu layar Anda "
            "hampir mencapai 8 jam sehari. Saran: Terapkan 'Digital Detox' secara berkala. Perbanyak aktivitas "
            "fisik atau kegiatan di luar ruangan untuk mencegah gaya hidup sedenter (kurang gerak) dan menjaga kesehatan mata."
        )
    else:
        return "Hubungi layanan dukungan mahasiswa."

# Simulasi pengujian untuk memastikan logika berjalan
for i in range(3):
    print(f"\nUser terdeteksi di Cluster {i}: \n{get_recommendation(i)}")

=== MEMULAI FASE 3: PERSONALIZED INTERVENTION (CLUSTERING) ===

[INFO] Model K-Means dan Scaler berhasil diekspor ke folder 'models/'.

=== HASIL EVALUASI CLUSTERING ===
Silhouette Coefficient : 0.2505 (Mendekati 1 lebih baik)

=== PROFIL BEHAVIORAL CLUSTER (RATA-RATA NILAI FITUR) ===
         Screen_Time  Assignment_Load  Sleep_Hours
Cluster                                           
0               7.82             5.23         4.91
1               2.59             4.50         6.53
2               7.90             5.34         8.14

=== SISTEM REKOMENDASI INTERVENSI PERSONAL ===

User terdeteksi di Cluster 0: 
Kategori: Kurang Tidur & Paparan Layar Tinggi
Intervensi: Jam tidur Anda sangat kurang (di bawah 5 jam) diiringi waktu layar yang tinggi. Saran: Kurangi menatap layar perangkat 1-2 jam sebelum tidur. Prioritaskan perbaikan 'Sleep Hygiene' agar mencapai minimal 6-7 jam tidur untuk menghindari kelelahan fisik (burnout).

User terdeteksi di Cluster 1: 
Kategori: Gaya Hidup Akadem